# Amazn AI - Day 2
## Embedding Generation & FAISS Vector Store

## Step 1: Import Libraries

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

## Step 2: Load Cleaned Dataset

In [ ]:
df=pd.read_csv('../data/processed/amazon_cleaned.csv')
df.head()

## Step 3: Load Embedding Model

In [ ]:
model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded successfully!')

## Step 4: Create Product Documents

In [ ]:
documents=[]
for _,row in df.iterrows():
    doc=f"""Product Name: {row['product_name']}

Category: {row['category']}

Discounted Price: ₹{row['discounted_price']}

Actual Price: ₹{row['actual_price']}

Discount: {row['discount_percentage']}%

Rating: {row['rating']}

Description:
{row['about_product']}
"""
    documents.append(doc)
print(len(documents))

## Step 5: Generate Embeddings

In [ ]:
embeddings=model.encode(documents,show_progress_bar=True)
print(embeddings.shape)

## Step 6: Create LangChain Documents

In [ ]:
langchain_docs=[]
for i,row in df.iterrows():
    langchain_docs.append(Document(page_content=documents[i],metadata={
        'chunk_id':i,
        'product_name':row['product_name'],
        'category':row['category'],
        'discounted_price':row['discounted_price'],
        'actual_price':row['actual_price'],
        'discount_percentage':row['discount_percentage'],
        'rating':row['rating'],
        'rating_count':row['rating_count']
    }))
print(len(langchain_docs))

## Step 7: Build and Save FAISS

In [ ]:
embedding_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(langchain_docs,embedding_model)
print(vector_store.index.ntotal)
vector_store.save_local('../vector_store/faiss_index')

## Step 8: Test Retrieval

In [ ]:
query='Laptop with 16GB RAM and 512GB SSD'
results=vector_store.similarity_search(query,k=5)
for i,r in enumerate(results):
    print('='*50)
    print(i+1,r.metadata['product_name'])
    print(r.metadata['category'])
    print(r.metadata['discounted_price'])
    print(r.metadata['rating'])

# Day 2 Complete
Generated embeddings, created FAISS vector store, and tested semantic search.